# M07 · Vecteurs et géométrie de ℝⁿ — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m07_vecteurs_rn/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m07_vecteurs_rn/03_notebook.ipynb`, créée par `python tools/start_chapter.py M07`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M07.1 à M07.13, M07.19 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M07.24 | Normes Lp et similarité cosinus · `norm_p, cosine_similarity` | 🔨 | ★ | 20 |
|  | M07.25 | La projection sur une droite · `project_onto_line` | 🔨 | ★ | 15 |
|  | M07.26 | Barycentre et somme des carrés des distances · `centroid, sum_sq_distances_to` | 🔨 | ★ | 20 |
|  | M07.27 | Côté d'un hyperplan et distance à un hyperplan · `side_of_hyperplane, distance_to_hyperplane` | 🔨 | ★ | 20 |
|  | M07.28 | `np.linalg.norm` (ord = 1, 2, inf) et `scipy.spatial.distance` contre tes fonctions | 📦 | ★ | 10 |
| B | M07.29 | Prédire : l'angle de deux vecteurs tirés au hasard en dimension 1 000 | 🔮 | ★ | 10 |
|  | M07.30 | La norme d'un vecteur gaussien se concentre autour de √d | 🔬 | ★★ | 20 |
|  | M07.31 | Médiane ou moyenne : le centre qui minimise les distances L1 ou leurs carrés, face à des points aberrants | 🔬 | ★★ | 20 |
|  | M07.32 | Les boules unités L1, L2, L∞ et Lp dans le plan | 🎨 | ★ | 20 |
| C | M07.33 | Chasse à l'erreur : une distance sans racine, un cosinus mal normalisé, une projection sur un vecteur non unitaire | 🐛 | ★ | 15 |
|  | M07.34 | Les normes de toutes les lignes d'un tableau : axis, keepdims et broadcasting | 🛠️ | ★ | 15 |
|  | M07.35 | Défi : le point qui minimise la somme des distances (non au carré) à 60 villes, à 0,1 % près | 🏆 | ★★★ | 30 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M07")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : combinaisons linéaires et one-hot, produits scalaires, normes et distances, cosinus, projection, hyperplans, score linéaire, barycentre et inertie, silhouette, k-means++, diagonale du cube, distance moyenne dans le cube, unités et plus proche voisin.
- Écrire les sept fonctions de `mymath/vectors.py` et les tester, puis comparer avec NumPy et SciPy.
- Prédire l'angle de deux vecteurs aléatoires, mesurer la concentration de la norme, comparer médiane et moyenne, tracer les boules unités, chasser des erreurs, vectoriser des normes, trouver une médiane géométrique.

**Rappel express.** $\mathbf{u} \cdot \mathbf{v} = \sum u_i v_i$ ; $\lVert \mathbf{x} \rVert_1 = \sum |x_i|$, $\lVert \mathbf{x} \rVert_2 = \sqrt{\sum x_i^2}$, $\lVert \mathbf{x} \rVert_\infty = \max |x_i|$ ; $\cos\theta = \frac{\mathbf{u} \cdot \mathbf{v}}{\lVert \mathbf{u} \rVert\,\lVert \mathbf{v} \rVert}$ ; projection $\frac{\mathbf{x} \cdot \mathbf{u}}{\mathbf{u} \cdot \mathbf{u}}\,\mathbf{u}$ ; distance à l'hyperplan $\frac{|\mathbf{w} \cdot \mathbf{x} + b|}{\lVert \mathbf{w} \rVert}$ ; $\sum \lVert \mathbf{x}_i - \mathbf{c} \rVert^2 = \sum \lVert \mathbf{x}_i - \bar{\mathbf{x}} \rVert^2 + m\,\lVert \bar{\mathbf{x}} - \mathbf{c} \rVert^2$.

## Partie 0 · Vérifier tes exercices papier (✏️ M07.1 à M07.13, 🔗 M07.19)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une fraction irréductible entre guillemets, `"7/9"` ;
- une liste `[-1.5, 2.5, 0]` (un vecteur, un point, une liste de valeurs), `True` / `False`, ou une lettre entre guillemets, `"K"`.

Les réponses pas encore remplies affichent ⏳.

In [ ]:
from fractions import Fraction   # Fraction(3, 4) is accepted too, as "3/4"

**Ex M07.1 — Combinaisons linéaires et vecteurs one-hot dans ℝ⁴**

In [ ]:
# Ex M07.1: the values you found on paper
answer_M07_1a = ...  # a) a list of 4 integers
answer_M07_1b = ...  # b) a list of 4 integers
answer_M07_1c = ...  # c) a list of 4 integers
answer_M07_1d = ...  # d) the list [a, b]
answer_M07_1e = ...  # e) a list of 4 integers
answer_M07_1f = ...  # f) a list of 4 integers
answer_M07_1g = ...  # g) a list of 4 numbers

for letter, answer in zip("abcdefg", [answer_M07_1a, answer_M07_1b, answer_M07_1c, answer_M07_1d, answer_M07_1e, answer_M07_1f, answer_M07_1g]):
    wbm.check(f"M07.1{letter}", answer)

**Ex M07.2 — Produits scalaires, ‖u + v‖² et l'identité du parallélogramme**

In [ ]:
# Ex M07.2: the values you found on paper
answer_M07_2a = ...  # a) an integer
answer_M07_2b = ...  # b) an integer
answer_M07_2c = ...  # c) an integer
answer_M07_2d = ...  # d) an integer
answer_M07_2e = ...  # e) an irreducible fraction
answer_M07_2f = ...  # f) an integer
answer_M07_2g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M07_2a, answer_M07_2b, answer_M07_2c, answer_M07_2d, answer_M07_2e, answer_M07_2f, answer_M07_2g]):
    wbm.check(f"M07.2{letter}", answer)

**Ex M07.3 — Normes L1, L2, L∞ et distances entre trois profils**

In [ ]:
# Ex M07.3: the values you found on paper
answer_M07_3a = ...  # a) a list of 3 numbers
answer_M07_3b = ...  # b) a list of 3 numbers
answer_M07_3c = ...  # c) a number (3 decimals)
answer_M07_3d = ...  # d) "Q" or "R"
answer_M07_3e = ...  # e) "Q" or "R"
answer_M07_3f = ...  # f) a list (4 decimals)
answer_M07_3g = ...  # g) a number (3 decimals)

for letter, answer in zip("abcdefg", [answer_M07_3a, answer_M07_3b, answer_M07_3c, answer_M07_3d, answer_M07_3e, answer_M07_3f, answer_M07_3g]):
    wbm.check(f"M07.3{letter}", answer)

**Ex M07.4 — Similarité cosinus entre des profils d'écoute musicale**

In [ ]:
# Ex M07.4: the values you found on paper
answer_M07_4a = ...  # a) an integer
answer_M07_4b = ...  # b) an irreducible fraction
answer_M07_4c = ...  # c) a number (2 decimals)
answer_M07_4d = ...  # d) a number (4 decimals)
answer_M07_4e = ...  # e) a number
answer_M07_4f = ...  # f) a number
answer_M07_4g = ...  # g) "C" or "E"
answer_M07_4h = ...  # h) "C" or "E"

for letter, answer in zip("abcdefgh", [answer_M07_4a, answer_M07_4b, answer_M07_4c, answer_M07_4d, answer_M07_4e, answer_M07_4f, answer_M07_4g, answer_M07_4h]):
    wbm.check(f"M07.4{letter}", answer)

**Ex M07.5 — Projeter sur une droite : composante parallèle et composante orthogonale**

In [ ]:
# Ex M07.5: the values you found on paper
answer_M07_5a = ...  # a) an integer
answer_M07_5b = ...  # b) a number
answer_M07_5c = ...  # c) a list of 4 numbers
answer_M07_5d = ...  # d) a list of 4 numbers
answer_M07_5e = ...  # e) a number
answer_M07_5f = ...  # f) a number
answer_M07_5g = ...  # g) a list of 4 numbers
answer_M07_5h = ...  # h) a number

for letter, answer in zip("abcdefgh", [answer_M07_5a, answer_M07_5b, answer_M07_5c, answer_M07_5d, answer_M07_5e, answer_M07_5f, answer_M07_5g, answer_M07_5h]):
    wbm.check(f"M07.5{letter}", answer)

**Ex M07.6 — Un plan de ℝ³ : vecteur normal, côté, distance signée**

In [ ]:
# Ex M07.6: the values you found on paper
answer_M07_6a = ...  # a) a list of 3 integers
answer_M07_6b = ...  # b) True or False
answer_M07_6c = ...  # c) 1, -1 or 0
answer_M07_6d = ...  # d) a number
answer_M07_6e = ...  # e) an irreducible fraction
answer_M07_6f = ...  # f) a number
answer_M07_6g = ...  # g) a list (4 decimals)
answer_M07_6h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M07_6a, answer_M07_6b, answer_M07_6c, answer_M07_6d, answer_M07_6e, answer_M07_6f, answer_M07_6g, answer_M07_6h]):
    wbm.check(f"M07.6{letter}", answer)

**Ex M07.7 — Un score linéaire w · x + b : calculer, décider, lire le vecteur normal**

In [ ]:
# Ex M07.7: the values you found on paper
answer_M07_7a = ...  # a) a number
answer_M07_7b = ...  # b) a number
answer_M07_7c = ...  # c) 1 or 0
answer_M07_7d = ...  # d) a number
answer_M07_7e = ...  # e) a number
answer_M07_7f = ...  # f) a list (3 decimals)
answer_M07_7g = ...  # g) a number (4 decimals)

for letter, answer in zip("abcdefg", [answer_M07_7a, answer_M07_7b, answer_M07_7c, answer_M07_7d, answer_M07_7e, answer_M07_7f, answer_M07_7g]):
    wbm.check(f"M07.7{letter}", answer)

**Ex M07.8 — Centroïde, inertie et formule de König-Huygens**

In [ ]:
# Ex M07.8: the values you found on paper
answer_M07_8a = ...  # a) a list of 2 numbers
answer_M07_8b = ...  # b) a number
answer_M07_8c = ...  # c) a number
answer_M07_8d = ...  # d) a number
answer_M07_8e = ...  # e) a number
answer_M07_8f = ...  # f) a list of 2 numbers
answer_M07_8g = ...  # g) a number

for letter, answer in zip("abcdefg", [answer_M07_8a, answer_M07_8b, answer_M07_8c, answer_M07_8d, answer_M07_8e, answer_M07_8f, answer_M07_8g]):
    wbm.check(f"M07.8{letter}", answer)

**Ex M07.9 — Où placer un dépôt : le barycentre pondéré des livraisons**

In [ ]:
# Ex M07.9: the values you found on paper
answer_M07_9a = ...  # a) an integer
answer_M07_9b = ...  # b) a list (4 decimals)
answer_M07_9c = ...  # c) an irreducible fraction
answer_M07_9d = ...  # d) an irreducible fraction
answer_M07_9e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_M07_9a, answer_M07_9b, answer_M07_9c, answer_M07_9d, answer_M07_9e]):
    wbm.check(f"M07.9{letter}", answer)

**Ex M07.10 — La silhouette d'un point, à la main, en dimension 2**

In [ ]:
# Ex M07.10: the values you found on paper
answer_M07_10a = ...  # a) a number (4 decimals)
answer_M07_10b = ...  # b) a number (4 decimals)
answer_M07_10c = ...  # c) a number (3 decimals)
answer_M07_10d = ...  # d) a number (3 decimals)
answer_M07_10e = ...  # e) a number (3 decimals)
answer_M07_10f = ...  # f) a number

for letter, answer in zip("abcdef", [answer_M07_10a, answer_M07_10b, answer_M07_10c, answer_M07_10d, answer_M07_10e, answer_M07_10f]):
    wbm.check(f"M07.10{letter}", answer)

**Ex M07.11 — k-means++ : des probabilités de tirage proportionnelles à D²**

In [ ]:
# Ex M07.11: the values you found on paper
answer_M07_11a = ...  # a) a list of 5 integers
answer_M07_11b = ...  # b) an irreducible fraction
answer_M07_11c = ...  # c) an irreducible fraction
answer_M07_11d = ...  # d) a list of 5 integers
answer_M07_11e = ...  # e) an irreducible fraction
answer_M07_11f = ...  # f) an irreducible fraction

for letter, answer in zip("abcdef", [answer_M07_11a, answer_M07_11b, answer_M07_11c, answer_M07_11d, answer_M07_11e, answer_M07_11f]):
    wbm.check(f"M07.11{letter}", answer)

**Ex M07.12 — La diagonale du cube unité et son angle avec les axes, quand d grandit**

In [ ]:
# Ex M07.12: the values you found on paper
answer_M07_12a = ...  # a) a number
answer_M07_12b = ...  # b) a number (4 decimals)
answer_M07_12c = ...  # c) a number
answer_M07_12d = ...  # d) a number (2 decimals)
answer_M07_12e = ...  # e) an integer
answer_M07_12f = ...  # f) an irreducible fraction

for letter, answer in zip("abcdef", [answer_M07_12a, answer_M07_12b, answer_M07_12c, answer_M07_12d, answer_M07_12e, answer_M07_12f]):
    wbm.check(f"M07.12{letter}", answer)

**Ex M07.13 — La distance au carré moyenne de deux points tirés dans le cube [0, 1]ᵈ**

In [ ]:
# Ex M07.13: the values you found on paper
answer_M07_13a = ...  # a) an irreducible fraction
answer_M07_13b = ...  # b) an irreducible fraction
answer_M07_13c = ...  # c) an irreducible fraction
answer_M07_13d = ...  # d) a number
answer_M07_13e = ...  # e) a number
answer_M07_13f = ...  # f) a number (3 decimals)

for letter, answer in zip("abcdef", [answer_M07_13a, answer_M07_13b, answer_M07_13c, answer_M07_13d, answer_M07_13e, answer_M07_13f]):
    wbm.check(f"M07.13{letter}", answer)

**Ex M07.19 — Pont : changer l'unité d'une feature change le plus proche voisin**

In [ ]:
# Ex M07.19: the values you found on paper
answer_M07_19a = ...  # a) a number (2 decimals)
answer_M07_19b = ...  # b) a number (2 decimals)
answer_M07_19c = ...  # c) "A" or "B"
answer_M07_19d = ...  # d) "A" or "B"
answer_M07_19e = ...  # e) a list of 2 numbers (3 decimals)
answer_M07_19f = ...  # f) "A" or "B"

for letter, answer in zip("abcdef", [answer_M07_19a, answer_M07_19b, answer_M07_19c, answer_M07_19d, answer_M07_19e, answer_M07_19f]):
    wbm.check(f"M07.19{letter}", answer)

## Partie A · Ta librairie : normes, cosinus, projection, barycentre, hyperplans ; NumPy et SciPy

Fiche §3 à §6. Les sept fonctions de `mymath/vectors.py`, puis les mêmes calculs avec NumPy et SciPy.

In [ ]:
# Tools of the notebook (parts A to C)
import math

import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial import distance as spd

from wbm.nbtools import checked, drawn, run_pytest_group


def raised(function, *args, **kwargs):
    """The name of the exception that function(*args, **kwargs) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳; so do the errors of a code
    that cannot run (a missing import, a misspelt name such as ValueErorr), for the check to name them."""
    try:
        function(*args, **kwargs)
    except (NotImplementedError, NameError, AttributeError, ImportError):
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"

### Ex M07.24 — Normes Lp et similarité cosinus · `norm_p, cosine_similarity` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer les normes $L^p$, L∞ comprise, et la similarité cosinus, en refusant les cas où elles n'ont pas de sens.  
**Prérequis :** fiche §3 et §4 · **mymath :** `vectors.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/vectors.py`, lis la docstring du module puis celle de la fonction, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Un vecteur est une liste ou un tableau NumPy à une dimension ; un ensemble de points, un tableau à deux dimensions, un point par ligne. NumPy suffit (sommes, `np.abs`, `np.sqrt`) ; `np.linalg.norm` et `scipy.spatial` sont interdits : les tests comparent tes résultats aux leurs, et un test vérifie que tu ne t'en sers pas. Les nombres renvoyés sont des `float` de Python, les vecteurs des tableaux NumPy.

Écris (fiche §3 et §4) :
- `norm_p(x, p=2)` : $\big(\sum_i |x_i|^p\big)^{1/p}$, et $\max_i |x_i|$ pour `p = float("inf")` ; une `ValueError` si `x` est vide ou si $p < 1$ (la formule n'est alors pas une norme) ;
- `cosine_similarity(u, v)` : $\frac{\mathbf{u} \cdot \mathbf{v}}{\lVert \mathbf{u} \rVert\,\lVert \mathbf{v} \rVert}$, ramené dans $[-1, 1]$ par `np.clip` (un arrondi peut donner 1,0000000000000002) ; une `ValueError` si les longueurs diffèrent ou si un vecteur est nul.

La vérification calcule, pour $\mathbf{x} = (1, -2, 2, -4)$ :
- a) `norm_p(x, 1)` ;
- b) `norm_p(x)` ;
- c) `norm_p(x, float("inf"))` ;
- d) `norm_p(x, 3)` (3 décimales) ;
- e) le nom de l'erreur levée par `norm_p([1, 2], 0.5)` ;

puis :
- f) `cosine_similarity([1, 2, 0, -1], [2, 1, 1, 0])` (4 décimales) ;
- g) le nom de l'erreur levée par `cosine_similarity([0, 0, 0], [1, 2, 3])` ;
- h) le nom du type de `norm_p(np.array([1.5, 2.0]), 3)`, en texte.

Dans ta copie : calcule a) à d) et f) à la main. Pourquoi la norme diminue-t-elle de a) à c) quand $p$ augmente ? Et pourquoi faut-il `np.clip` (essaie `cosine_similarity` d'un vecteur avec lui-même, sans lui) ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M07")  # reload your saved file

In [ ]:
X_24 = [1, -2, 2, -4]
with checked("M07.24a"):
    wbm.check("M07.24a", mymath.vectors.norm_p(X_24, 1), computed=True)
with checked("M07.24b"):
    wbm.check("M07.24b", mymath.vectors.norm_p(X_24), computed=True)
with checked("M07.24c"):
    wbm.check("M07.24c", mymath.vectors.norm_p(X_24, float("inf")), computed=True)
with checked("M07.24d"):
    wbm.check("M07.24d", mymath.vectors.norm_p(X_24, 3), computed=True)
with checked("M07.24e"):
    wbm.check("M07.24e", raised(mymath.vectors.norm_p, [1, 2], 0.5), computed=True)
with checked("M07.24f"):
    wbm.check("M07.24f", mymath.vectors.cosine_similarity([1, 2, 0, -1], [2, 1, 1, 0]), computed=True)
with checked("M07.24g"):
    wbm.check("M07.24g", raised(mymath.vectors.cosine_similarity, [0, 0, 0], [1, 2, 3]), computed=True)
with checked("M07.24h"):
    wbm.check("M07.24h", type(mymath.vectors.norm_p(np.array([1.5, 2.0]), 3)).__name__, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m07_vectors.py", "test_norm_p_ or test_cosine_similarity_ or test_vectors_uses_")

### Ex M07.25 — La projection sur une droite · `project_onto_line` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer la projection orthogonale sur une droite dirigée par un vecteur quelconque, et vérifier que le reste est orthogonal.  
**Prérequis :** fiche §4 ; Ex M07.24 · **mymath :** `vectors.py` · **Parcours :** C, E

Écris `project_onto_line(x, u)` (fiche §4) : $\frac{\mathbf{x} \cdot \mathbf{u}}{\mathbf{u} \cdot \mathbf{u}}\,\mathbf{u}$, en tableau NumPy ; une `ValueError` si les longueurs diffèrent ou si $\mathbf{u}$ est nul. Le vecteur $\mathbf{u}$ n'a pas besoin d'être unitaire.

La vérification calcule :
- a) `project_onto_line([3, -1, 2], [1, 1, 1])` (une liste, 4 décimales) ;
- b) le produit scalaire du reste $\mathbf{x} - \mathbf{p}$ de a) avec $\mathbf{u} = (1, 1, 1)$ (4 décimales) ;
- c) `project_onto_line([2, 5], [0, -3])` (une liste) ;
- d) `project_onto_line([4, 1, 0, -2], [2, 0, 0, 0])` (une liste) ;
- e) le nom de l'erreur levée par `project_onto_line([1, 2], [0, 0])` ;
- f) le nom du type de `project_onto_line([1, 2], [1, 1])`, en texte.

Dans ta copie : calcule a), c) et d) à la main. Que fait la projection sur un axe (regarde c) et d)) ? Pourquoi le signe de $\mathbf{u} = (0, -3)$ ne change-t-il rien en c) ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M07")  # reload your saved file

In [ ]:
with checked("M07.25a"):
    wbm.check("M07.25a", np.asarray(mymath.vectors.project_onto_line([3, -1, 2], [1, 1, 1])).tolist(),
              computed=True)
with checked("M07.25b"):
    p_25 = np.asarray(mymath.vectors.project_onto_line([3, -1, 2], [1, 1, 1]), dtype=float)
    wbm.check("M07.25b", float((np.array([3, -1, 2]) - p_25) @ np.array([1, 1, 1])), computed=True)
with checked("M07.25c"):
    wbm.check("M07.25c", np.asarray(mymath.vectors.project_onto_line([2, 5], [0, -3])).tolist(), computed=True)
with checked("M07.25d"):
    wbm.check("M07.25d", np.asarray(mymath.vectors.project_onto_line([4, 1, 0, -2], [2, 0, 0, 0])).tolist(),
              computed=True)
with checked("M07.25e"):
    wbm.check("M07.25e", raised(mymath.vectors.project_onto_line, [1, 2], [0, 0]), computed=True)
with checked("M07.25f"):
    wbm.check("M07.25f", type(mymath.vectors.project_onto_line([1, 2], [1, 1])).__name__, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m07_vectors.py", "test_project_onto_line_ or test_vectors_uses_")

### Ex M07.26 — Barycentre et somme des carrés des distances · `centroid, sum_sq_distances_to` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer le barycentre, pondéré ou non, et l'inertie d'un nuage autour d'un point, et vérifier la formule de König-Huygens.  
**Prérequis :** fiche §6 · **mymath :** `vectors.py` · **Parcours :** C, E

Écris (fiche §6) :
- `centroid(points, weights=None)` : la moyenne des lignes de `points`, ou $\frac{\sum_i \omega_i \mathbf{x}_i}{\sum_i \omega_i}$ avec des poids ; une `ValueError` si `points` est vide, ou si les poids n'ont pas la bonne longueur, ont une valeur négative ou une somme nulle ;
- `sum_sq_distances_to(points, c)` : $\sum_i \lVert \mathbf{x}_i - \mathbf{c} \rVert^2$, en `float` ; une `ValueError` si `c` n'a pas une coordonnée par colonne.

La vérification calcule, pour les quatre points `[[1, 4], [3, 0], [-1, 2], [5, 6]]` :
- a) `centroid(points)` (une liste) ;
- b) `centroid(points, weights=[1, 1, 2, 0])` (une liste) ;
- c) `sum_sq_distances_to(points, centroid(points))` ;
- d) `sum_sq_distances_to(points, [0, 0])` ;
- e) le nom de l'erreur levée par `centroid(points, weights=[1, -1, 1, 1])` ;
- f) le nom du type de `sum_sq_distances_to(points, [0, 0])`, en texte.

Dans ta copie : calcule a) à d) à la main, et vérifie d) avec la formule de König-Huygens. Pourquoi le dernier point ne compte-t-il pas en b) ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M07")  # reload your saved file

In [ ]:
POINTS_26 = [[1, 4], [3, 0], [-1, 2], [5, 6]]
with checked("M07.26a"):
    wbm.check("M07.26a", np.asarray(mymath.vectors.centroid(POINTS_26)).tolist(), computed=True)
with checked("M07.26b"):
    wbm.check("M07.26b", np.asarray(mymath.vectors.centroid(POINTS_26, weights=[1, 1, 2, 0])).tolist(),
              computed=True)
with checked("M07.26c"):
    wbm.check("M07.26c", mymath.vectors.sum_sq_distances_to(POINTS_26, mymath.vectors.centroid(POINTS_26)),
              computed=True)
with checked("M07.26d"):
    wbm.check("M07.26d", mymath.vectors.sum_sq_distances_to(POINTS_26, [0, 0]), computed=True)
with checked("M07.26e"):
    wbm.check("M07.26e", raised(mymath.vectors.centroid, POINTS_26, weights=[1, -1, 1, 1]), computed=True)
with checked("M07.26f"):
    wbm.check("M07.26f", type(mymath.vectors.sum_sq_distances_to(POINTS_26, [0, 0])).__name__, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m07_vectors.py", "test_centroid_ or test_sum_sq_distances_to_ or test_vectors_uses_")

### Ex M07.27 — Côté d'un hyperplan et distance à un hyperplan · `side_of_hyperplane, distance_to_hyperplane` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer le côté d'un hyperplan et la distance, signée ou non, pour un point ou pour toutes les lignes d'un tableau.  
**Prérequis :** fiche §5 ; Ex M07.24 · **mymath :** `vectors.py` · **Parcours :** C, E

Écris (fiche §5) :
- `side_of_hyperplane(x, w, b)` : le signe de $\mathbf{w} \cdot \mathbf{x} + b$ (1, −1 ou 0), en `int` de Python pour un point (`x` à une dimension), en tableau d'entiers pour plusieurs points (une ligne par point) ;
- `distance_to_hyperplane(x, w, b, signed=False)` : $\frac{|\mathbf{w} \cdot \mathbf{x} + b|}{\lVert \mathbf{w} \rVert}$, sans la valeur absolue si `signed=True` ; un `float` pour un point, un tableau pour plusieurs ;
- dans les deux cas, une `ValueError` si $\mathbf{w}$ est nul ou si sa longueur ne correspond pas aux points.

Pour un tableau `X` (un point par ligne), `np.sum(X * w, axis=1)` donne tous les produits scalaires d'un coup (le broadcasting de 0A), et `np.sum(x * w, axis=-1)` marche pour un point comme pour un tableau.

La vérification prend l'hyperplan $x_1 - 2x_2 + 2x_3 + 1 = 0$ (`w = [1, -2, 2]`, `b = 1`) et les points `X = [[1, 1, 1], [0, 0, 0], [-3, 0, 0], [2, 3, 0]]`, et calcule :
- a) `side_of_hyperplane(X, w, b)` (une liste) ;
- b) `distance_to_hyperplane(X, w, b)` (une liste, 4 décimales) ;
- c) `distance_to_hyperplane(X, w, b, signed=True)` (une liste, 4 décimales) ;
- d) le nom du type de `side_of_hyperplane([1, 1, 1], w, b)`, en texte ;
- e) `distance_to_hyperplane([1, 1, 1], w, b)` (4 décimales) ;
- f) le nom de l'erreur levée par `distance_to_hyperplane([1, 2], [0, 0], 1)`.

Dans ta copie : calcule a) et b) à la main. À quelle distance de l'origine passe l'hyperplan, et de quel côté est l'origine ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M07")  # reload your saved file

In [ ]:
W_27, B_27, X_27 = [1, -2, 2], 1, [[1, 1, 1], [0, 0, 0], [-3, 0, 0], [2, 3, 0]]
with checked("M07.27a"):
    wbm.check("M07.27a", np.asarray(mymath.vectors.side_of_hyperplane(X_27, W_27, B_27)).tolist(), computed=True)
with checked("M07.27b"):
    wbm.check("M07.27b", np.asarray(mymath.vectors.distance_to_hyperplane(X_27, W_27, B_27)).tolist(), computed=True)
with checked("M07.27c"):
    wbm.check("M07.27c", np.asarray(mymath.vectors.distance_to_hyperplane(X_27, W_27, B_27, signed=True)).tolist(),
              computed=True)
with checked("M07.27d"):
    wbm.check("M07.27d", type(mymath.vectors.side_of_hyperplane([1, 1, 1], W_27, B_27)).__name__, computed=True)
with checked("M07.27e"):
    wbm.check("M07.27e", mymath.vectors.distance_to_hyperplane([1, 1, 1], W_27, B_27), computed=True)
with checked("M07.27f"):
    wbm.check("M07.27f", raised(mymath.vectors.distance_to_hyperplane, [1, 2], [0, 0], 1), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m07_vectors.py", "test_side_of_hyperplane_ or test_distance_to_hyperplane_ or test_vectors_uses_")

### Ex M07.28 — `np.linalg.norm` (ord = 1, 2, inf) et `scipy.spatial.distance` contre tes fonctions 📦 ★ ⏱️ 10 min
**Objectif :** calculer des normes et des distances avec NumPy et SciPy, lire ce que renvoie chaque fonction, et comparer avec ta librairie.  
**Prérequis :** fiche §3 et §4 ; Ex M07.24 · **Parcours :** C

En pratique, on ne réécrit pas les normes : NumPy et SciPy les fournissent.
- `np.linalg.norm(x, ord=p)` donne la norme $L^p$ d'un vecteur (`ord=1`, `ord=2`, `ord=np.inf`) ;
- `scipy.spatial.distance` (importé sous le nom `spd`) donne les distances entre deux points : `spd.euclidean(a, b)`, `spd.cityblock(a, b)` (L1, la distance de Manhattan), `spd.chebyshev(a, b)` (L∞), et `spd.cosine(a, b)`, qui renvoie une **distance cosinus**, $1 - \cos\theta$.

Avec $\mathbf{a} = (2, -1, 1, 3)$ et $\mathbf{b} = (-1, 1, 4, 1)$, calcule avec ces fonctions :
- a) `spd.cityblock(a, b)` ;
- b) `spd.chebyshev(a, b)` ;
- c) `spd.euclidean(a, b)` (4 décimales) ;
- d) `spd.cosine(a, b)` (4 décimales) ;
- e) `np.linalg.norm(a, ord=np.inf)` ;
- f) vrai ou faux : pour 20 vecteurs tirés au hasard et $p \in \{1, 2, 3, \infty\}$, ta fonction `norm_p` donne la même valeur que `np.linalg.norm` à $10^{-12}$ près (calcule-le avec une boucle et `np.isclose`).

Dans ta copie : quelle similarité cosinus correspond à d) ? Pourquoi SciPy préfère-t-il une distance, $1 - \cos\theta$, plutôt qu'une similarité ?

In [ ]:
answer_M07_28a = ...  # a) a number
answer_M07_28b = ...  # b) a number
answer_M07_28c = ...  # c) a number (4 decimals)
answer_M07_28d = ...  # d) a number (4 decimals)
answer_M07_28e = ...  # e) a number
answer_M07_28f = ...  # f) True or False, computed with a loop over random vectors

In [ ]:
for letter, answer in zip("abcdef", [answer_M07_28a, answer_M07_28b, answer_M07_28c, answer_M07_28d,
                                     answer_M07_28e, answer_M07_28f]):
    wbm.check(f"M07.28{letter}", answer)

## Partie B · Une prédiction, deux expériences et une figure

Fiche §3, §6 et §7. L'angle de deux vecteurs aléatoires, la norme d'un vecteur gaussien, la médiane face à la moyenne, les boules unités.

### Ex M07.29 — Prédire : l'angle de deux vecteurs tirés au hasard en dimension 1 000 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir la loi de l'angle entre deux vecteurs aléatoires selon la dimension, puis la mesurer.  
**Prérequis :** fiche §4 et §7 · **Parcours :** C, E

On tire deux vecteurs de $\mathbb{R}^d$ dont les coordonnées sont des variables indépendantes de loi $\mathcal{N}(0, 1)$, et l'on calcule l'angle qu'ils font. On recommence 2 000 fois. Choisis, pour chaque question :
- a) en dimension 1 000, l'angle est le plus souvent : (1) proche de 0° ; (2) proche de 45° ; (3) proche de 90° ; (4) n'importe où entre 0° et 180°, sans préférence ;
- b) en dimension 1 000, la part des tirages dont l'angle est entre 85° et 95° : (1) moins de 10 % ; (2) environ la moitié ; (3) plus de 99 % ;
- c) quand la dimension passe de 10 à 1 000, l'écart-type des angles : (1) grandit ; (2) diminue ; (3) reste le même ;
- d) en dimension 2, l'angle est : (1) réparti uniformément entre 0° et 180° ; (2) concentré près de 90° ; (3) concentré près de 0° et de 180°.

**Avant d'exécuter quoi que ce soit**, écris tes quatre prédictions (le **numéro du choix**), vérifie-les, puis lance l'expérience.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M07_29a = ...  # a) 1, 2, 3 or 4, written BEFORE running the experiment below
prediction_M07_29b = ...  # b) 1, 2 or 3
prediction_M07_29c = ...  # c) 1, 2 or 3
prediction_M07_29d = ...  # d) 1, 2 or 3

In [ ]:
for letter, prediction in zip("abcd", [prediction_M07_29a, prediction_M07_29b, prediction_M07_29c,
                                      prediction_M07_29d]):
    wbm.check(f"M07.29{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes quatre prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M07_29a, prediction_M07_29b, prediction_M07_29c, prediction_M07_29d]):
    print("⏳ Écris d'abord tes quatre prédictions, puis exécute à nouveau.")
else:
    rng_29 = np.random.default_rng(729)
    for d in (2, 10, 1000):
        X, Y = rng_29.normal(size=(2000, d)), rng_29.normal(size=(2000, d))
        cosines = np.sum(X * Y, axis=1) / (np.sqrt(np.sum(X ** 2, axis=1)) * np.sqrt(np.sum(Y ** 2, axis=1)))
        angles = np.degrees(np.arccos(np.clip(cosines, -1, 1)))
        print(f"d = {d:4d} · mean angle {angles.mean():6.2f}° · standard deviation {angles.std():6.2f}° · "
              f"share between 85° and 95°: {np.mean((angles > 85) & (angles < 95)):.3f}")
    fig, ax = plt.subplots(figsize=(6, 3.2))
    for d in (2, 10, 1000):
        X, Y = rng_29.normal(size=(2000, d)), rng_29.normal(size=(2000, d))
        cosines = np.sum(X * Y, axis=1) / (np.sqrt(np.sum(X ** 2, axis=1)) * np.sqrt(np.sum(Y ** 2, axis=1)))
        ax.hist(np.degrees(np.arccos(np.clip(cosines, -1, 1))), bins=np.linspace(0, 180, 73), histtype="step",
                density=True, lw=1.8, label=f"d = {d}")
    ax.set(xlabel="angle (degrés)", ylabel="densité", title="Angle de deux vecteurs gaussiens")
    ax.legend()
    plt.show()

**Après l'expérience**, dans ta copie : relie l'écart-type mesuré en dimension 1 000 au calcul de la fiche (§7 : le cosinus est de l'ordre de $\frac{1}{\sqrt{d}}$, et près de 90°, un petit cosinus $c$ correspond à un écart d'environ $c$ radians). Pourquoi la dimension 2 se comporte-t-elle si différemment ? Qu'en conclus-tu pour des *embeddings* de dimension 768 qui n'auraient aucune structure ?

### Ex M07.30 — La norme d'un vecteur gaussien se concentre autour de √d 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer comment la norme d'un vecteur gaussien se rapproche de $\sqrt{d}$, et comment sa dispersion relative diminue avec la dimension.  
**Prérequis :** fiche §7 ; M02b (loi normale) ; M02c (écart-type) · **Parcours :** C

La fiche (§7) annonce que la norme d'un vecteur $\mathbf{X}$ de $\mathbb{R}^d$ à coordonnées gaussiennes indépendantes vaut environ $\sqrt{d}$, avec une dispersion qui ne grandit pas. Vérifie-le sur les échantillons fixés de la cellule ci-dessous : 2 000 vecteurs dans chaque dimension 1, 10, 100 et 1 000.

Écris `norm_summary(X)`, qui reçoit un tableau de vecteurs (un par ligne, de dimension $d$) et renvoie le couple `(moyenne des normes divisée par √d, écart-type des normes)`, deux `float` (l'écart-type avec `np.std`, `ddof = 0`). Pas de boucle : les normes de toutes les lignes s'obtiennent d'un coup (`axis=1`).

La vérification calcule :
- a) la moyenne des normes divisée par $\sqrt{1000}$, en dimension 1 000 (2 décimales) ;
- b) l'écart-type des normes en dimension 1 000 (2 décimales) ;
- c) l'écart-type des normes en dimension 1 (2 décimales) ;
- d) la pente de la droite qui ajuste $\ln\big(\frac{\text{écart-type}}{\sqrt{d}}\big)$ en fonction de $\ln d$, pour les quatre dimensions (`np.polyfit`, 1 décimale).

Dans ta copie : que dit d) de la dispersion **relative** de la norme ? Compare c) à $\sqrt{1 - \frac{2}{\pi}} \approx 0{,}603$, l'écart-type exact de $|Z|$ pour $Z$ de loi $\mathcal{N}(0, 1)$. Vers quelle valeur l'écart-type de b) semble-t-il tendre ?

In [ ]:
rng_30 = np.random.default_rng(737)
DIMS_30 = [1, 10, 100, 1000]
SAMPLES_30 = {d: rng_30.normal(size=(2000, d)) for d in DIMS_30}      # 2,000 Gaussian vectors in each dimension

In [ ]:
def norm_summary(X):
    """(mean of the norms of the rows of X divided by √d, standard deviation of the norms), two Python floats."""
    raise NotImplementedError("les normes de toutes les lignes avec axis=1, puis np.mean et np.std")

In [ ]:
with checked("M07.30a"):
    wbm.check("M07.30a", norm_summary(SAMPLES_30[1000])[0], computed=True)
with checked("M07.30b"):
    wbm.check("M07.30b", norm_summary(SAMPLES_30[1000])[1], computed=True)
with checked("M07.30c"):
    wbm.check("M07.30c", norm_summary(SAMPLES_30[1])[1], computed=True)
with checked("M07.30d"):
    spreads_30 = [norm_summary(SAMPLES_30[d])[1] / math.sqrt(d) for d in DIMS_30]
    wbm.check("M07.30d", float(np.polyfit(np.log(DIMS_30), np.log(spreads_30), 1)[0]), computed=True)

### Ex M07.31 — Médiane ou moyenne : le centre qui minimise les distances L1 ou leurs carrés, face à des points aberrants 🔬 ★★ ⏱️ 20 min
**Objectif :** comparer le point qui minimise la somme des distances L1 (la médiane, coordonnée par coordonnée) à celui qui minimise la somme des carrés des distances (le barycentre), et leur résistance aux points aberrants.  
**Prérequis :** fiche §3 et §6 ; Ex M07.26 · **Parcours :** C

Le barycentre minimise $\sum_i \lVert \mathbf{x}_i - \mathbf{c} \rVert_2^2$ (fiche §6). Et la somme des distances L1, $\sum_i \lVert \mathbf{x}_i - \mathbf{c} \rVert_1$ ? Elle se découpe coordonnée par coordonnée : $\sum_i \lVert \mathbf{x}_i - \mathbf{c} \rVert_1 = \sum_j \sum_i |x_{ij} - c_j|$, et chaque somme $\sum_i |x_{ij} - c_j|$ est la plus petite quand $c_j$ est une **médiane** des $x_{ij}$ (pour un nombre impair de valeurs, la valeur du milieu). Le point qui minimise la somme des distances L1 est donc la médiane, coordonnée par coordonnée.

Écris `l1_center(points)`, qui renvoie ce point en tableau NumPy (`np.median` avec le bon `axis`), et `l1_cost(points, c)`, qui renvoie $\sum_i \lVert \mathbf{x}_i - \mathbf{c} \rVert_1$ en `float`.

Les neuf points `[[1, 2], [2, 3], [3, 1], [2, 2], [4, 3], [3, 4], [1, 1], [5, 2], [2, 5]]` décrivent les positions d'un petit troupeau ; deux points aberrants `[[40, 2], [38, 1]]` y sont ajoutés (une erreur de saisie). La vérification calcule, avec ta fonction `mymath.vectors.centroid` pour le barycentre (écris d'abord M07.26) :
- a) `l1_center` des neuf points (une liste) ;
- b) leur barycentre (une liste, 4 décimales) ;
- c) `l1_center` des onze points (une liste) ;
- d) le barycentre des onze points (une liste, 4 décimales) ;
- e) la distance euclidienne dont le barycentre s'est déplacé entre b) et d) (4 décimales) ;
- f) la distance euclidienne dont la médiane s'est déplacée entre a) et c) ;
- g) le rapport `l1_cost(onze points, barycentre) / l1_cost(onze points, médiane)` (4 décimales).

Dans ta copie : pourquoi la médiane résiste-t-elle aux points aberrants, et pas le barycentre ? Lequel des deux décrit le mieux le troupeau ? Que confirme g) ?

In [ ]:
def l1_center(points):
    """The coordinate-wise median of the rows of points: the point that minimises the sum of the L1 distances."""
    raise NotImplementedError("np.median, colonne par colonne")


def l1_cost(points, c):
    """The sum of the L1 distances from the rows of points to c, as a Python float."""
    raise NotImplementedError("les valeurs absolues des écarts, puis la somme")

In [ ]:
POINTS_31, OUTLIERS_31 = np.array([[1, 2], [2, 3], [3, 1], [2, 2], [4, 3], [3, 4], [1, 1], [5, 2], [2, 5]]), np.array([[40, 2], [38, 1]])
ALL_31 = np.vstack([POINTS_31, OUTLIERS_31])
with checked("M07.31a"):
    wbm.check("M07.31a", np.asarray(l1_center(POINTS_31), dtype=float).tolist(), computed=True)
with checked("M07.31b"):
    wbm.check("M07.31b", np.asarray(mymath.vectors.centroid(POINTS_31)).tolist(), computed=True)
with checked("M07.31c"):
    wbm.check("M07.31c", np.asarray(l1_center(ALL_31), dtype=float).tolist(), computed=True)
with checked("M07.31d"):
    wbm.check("M07.31d", np.asarray(mymath.vectors.centroid(ALL_31)).tolist(), computed=True)
with checked("M07.31e"):
    moved_mean = np.asarray(mymath.vectors.centroid(ALL_31)) - np.asarray(mymath.vectors.centroid(POINTS_31))
    wbm.check("M07.31e", float(np.sqrt(np.sum(moved_mean ** 2))), computed=True)
with checked("M07.31f"):
    moved_median = np.asarray(l1_center(ALL_31), dtype=float) - np.asarray(l1_center(POINTS_31), dtype=float)
    wbm.check("M07.31f", float(np.sqrt(np.sum(moved_median ** 2))), computed=True)
with checked("M07.31g"):
    wbm.check("M07.31g", l1_cost(ALL_31, mymath.vectors.centroid(ALL_31)) / l1_cost(ALL_31, l1_center(ALL_31)),
              computed=True)

### Ex M07.32 — Les boules unités L1, L2, L∞ et Lp dans le plan 🎨 ★ ⏱️ 20 min
**Objectif :** tracer les vecteurs de norme 1 pour plusieurs normes $L^p$, en normalisant des directions, et voir les boules s'emboîter.  
**Prérequis :** fiche §3 ; R0 (matplotlib) · **Parcours :** C

Écris `draw_unit_circle(ax, p, **style)`, qui trace dans l'axe `ax` la courbe des vecteurs du plan de norme $L^p$ égale à 1. Une méthode simple : prends 400 directions $\mathbf{d}_k = (\cos t_k, \sin t_k)$ régulièrement réparties de 0 à $2\pi$, divise chacune par sa norme $L^p$ (ta fonction `norm_p`, ou la formule, avec `float("inf")` pour L∞), et relie les points obtenus par `ax.plot(xs, ys, **style)`. Les arguments `**style` (couleur, épaisseur, légende) passent tels quels à `ax.plot`.

Puis trace sur la même figure les courbes de $p = 1$, $1{,}5$, $2$, $4$ et $\infty$, avec une légende, la même échelle sur les deux axes (`ax.set_aspect("equal")`) et les axes de coordonnées. Ajoute, en pointillés, la courbe de $p = \frac{1}{2}$ : pour elle, `norm_p` lève une erreur, utilise la formule.

Une cellule de vérification essaie ta fonction pour cinq valeurs de $p$ (la figure elle-même se juge à l'œil).

Dans ta copie : comment les courbes s'emboîtent-elles, et quelle inégalité entre normes cela traduit-il ? Pourquoi la courbe de $p = \frac{1}{2}$ montre-t-elle, d'un seul coup d'œil, que ce n'est pas une norme (pense au segment qui joint $(1, 0)$ à $(0, 1)$) ?

In [ ]:
def draw_unit_circle(ax, p, **style):
    """In ax, the curve of the vectors of the plane whose L^p norm equals 1 (p >= 1 or float("inf"); the formula
    also for 0 < p < 1), drawn with ax.plot(..., **style)."""
    raise NotImplementedError("400 directions (cos t, sin t), chacune divisée par sa norme L^p, puis ax.plot")

In [ ]:
with checked("M07.32 figure"):
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    try:
        for p, style in [(1, {}), (1.5, {}), (2, {}), (4, {}), (float("inf"), {}), (0.5, {"ls": "--", "color": "gray"})]:
            created = drawn(lambda p=p, style=style: draw_unit_circle(ax, p, label=f"p = {p}", **style))
    except BaseException:
        plt.close(fig)                                     # no empty figure while the function is not written yet
        raise
    if created:
        plt.close(fig)
        print("❌ Ex M07.32 : draw_unit_circle ouvre sa propre figure : dessine dans l'axe reçu.")
    else:
        ax.axhline(0, color="black", lw=0.5)
        ax.axvline(0, color="black", lw=0.5)
        ax.set_aspect("equal")
        ax.legend(fontsize=8)
        plt.show()

**Vérifications automatiques** (sur cinq valeurs de $p$, chacune dans un axe neuf).

In [ ]:
with checked("M07.32 vérifications"):
    import matplotlib.lines as mlines
    problems = []
    for p_t in (1, 1.5, 2, 3, float("inf")):
        fig_t, ax_t = plt.subplots()
        try:
            created = drawn(draw_unit_circle, ax_t, p_t)
        except NotImplementedError:
            plt.close(fig_t)
            raise
        lines_t = [line for line in ax_t.get_lines() if len(line.get_xdata()) > 1]
        plt.close(fig_t)
        if created:
            problems.append("draw_unit_circle ouvre sa propre figure (plt.figure ou plt.subplots) : dessine dans l'axe "
                            "reçu")
            break
        if not lines_t:
            problems.append(f"pour p = {p_t}, aucune courbe n'est tracée : ax.plot(xs, ys)")
            break
        xy = np.concatenate([np.column_stack([line.get_xdata(), line.get_ydata()]) for line in lines_t])
        xy = xy[np.isfinite(xy).all(axis=1)]
        norms_t = (np.max(np.abs(xy), axis=1) if math.isinf(p_t)
                   else np.sum(np.abs(xy) ** p_t, axis=1) ** (1 / p_t))
        angles_t = np.arctan2(xy[:, 1], xy[:, 0])
        if len(xy) < 100:
            problems.append(f"pour p = {p_t}, la courbe a {len(xy)} points : prends-en au moins 100 pour qu'elle soit "
                            "lisse")
            break
        if not np.allclose(norms_t, 1, atol=1e-6):
            problems.append(f"pour p = {p_t}, des points tracés n'ont pas une norme L{p_t} égale à 1 (de "
                            f"{norms_t.min():.3f} à {norms_t.max():.3f}) : divise chaque direction par sa norme")
            break
        if np.histogram(angles_t, bins=8, range=(-np.pi, np.pi))[0].min() == 0:
            problems.append(f"pour p = {p_t}, la courbe ne fait pas tout le tour de l'origine : prends des directions "
                            "sur tout le cercle, de 0 à 2π")
            break
    if problems:
        print("❌ Ex M07.32 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M07.32 : draw_unit_circle trace des courbes fermées dont tous les points ont la norme 1, pour "
              "cinq valeurs de p.")

## Partie C · Erreurs classiques, savoir-faire et défi

Fiche §1, §3, §4 et §6. Trois erreurs à trouver, les normes de toutes les lignes d'un tableau, puis la médiane géométrique de 60 villes.

### Ex M07.33 — Chasse à l'erreur : une distance sans racine, un cosinus mal normalisé, une projection sur un vecteur non unitaire 🐛 ★ ⏱️ 15 min
**Objectif :** repérer dans du code les erreurs classiques sur les distances, les cosinus et les composantes, puis calculer les bonnes valeurs.  
**Prérequis :** fiche §3 et §4 · **Parcours :** C

Un collègue a écrit six lignes, chacune avec un commentaire qui dit ce qu'elle doit calculer, pour $\mathbf{a} = (1, 3, -2)$ et $\mathbf{b} = (4, -1, 2)$. Trois lignes ne calculent pas ce que dit leur commentaire. **Sans exécuter de code**, trouve-les.

```python
a, b = np.array([1.0, 3.0, -2.0]), np.array([4.0, -1.0, 2.0])
dist = np.sum((a - b) ** 2)                         # (A) the Euclidean distance between a and b
dist_l1 = np.sum(np.abs(a - b))                     # (B) the L1 distance between a and b
cos = (a @ b) / np.sqrt(a @ a)                      # (C) the cosine similarity of a and b
norm_b = np.sqrt(b @ b)                             # (D) the Euclidean norm of b
comp = (b @ a) / (a @ a)                            # (E) the component of b along a (a signed length)
dist_inf = np.max(np.abs(a - b))                    # (F) the L∞ distance between a and b
```

Questions :
- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;
- b) pour chacune des six lignes, de (A) à (F), la bonne valeur de ce que dit son commentaire (une liste de 6 nombres, 3 décimales).

Dans ta copie : pour chaque ligne fausse, quelle erreur a été faite, et comment la corriger ? La ligne (E) calcule autre chose qu'une composante : quoi, et à quoi sert ce nombre (fiche §4) ?

In [ ]:
answer_M07_33a = ...  # a) letters, as text
answer_M07_33b = ...  # b) a list of 6 values (3 decimals)

In [ ]:
wbm.check("M07.33a", answer_M07_33a)
wbm.check("M07.33b", answer_M07_33b)

### Ex M07.34 — Les normes de toutes les lignes d'un tableau : axis, keepdims et broadcasting 🛠️ ★ ⏱️ 15 min
**Objectif :** calculer les normes de toutes les lignes d'un tableau et normaliser chaque ligne d'un coup, sans boucle, avec `axis` et `keepdims`.  
**Prérequis :** fiche §1 et §3 ; 0A (broadcasting) · **Parcours :** C

Un dataset est un tableau `X` de forme `(n, p)`, un exemple par ligne. Calculer la norme de chaque exemple, ou le normaliser, se fait d'un coup avec NumPy :
- `np.sum(A, axis=1)` additionne chaque **ligne** (on « écrase » l'axe des colonnes) et rend un tableau de forme `(n,)` ;
- avec `keepdims=True`, le résultat garde la forme `(n, 1)` : une colonne, qui se divise ligne par ligne par le **broadcasting** (`X / colonne` divise chaque ligne par son nombre).

Écris, **sans boucle** :
- `row_norms(X, p=2)`, qui renvoie le tableau des normes $L^p$ des lignes de `X` (`p = float("inf")` pour L∞) ;
- `normalize_rows(X)`, qui renvoie le tableau de même forme dont chaque ligne est divisée par sa norme L2 ; une ligne nulle reste nulle (pas de division par zéro : `np.where`).

Une cellule de vérification compare tes fonctions à une boucle (ligne nulle comprise) et vérifie qu'elles n'en contiennent pas ; une autre compare les temps de calcul.

Dans ta copie : que donnerait `np.sum(X ** 2, axis=0)` ? Pourquoi `X / row_norms(X)` (sans `keepdims`) échoue-t-il, ou se trompe-t-il quand `X` est carré ? Combien de fois plus rapide est `row_norms` que la boucle ?

In [ ]:
def row_norms(X, p=2):
    """The L^p norms of the rows of X (p = float("inf") for L∞), without a Python loop."""
    raise NotImplementedError("np.abs, puis une somme (ou un maximum) sur axis=1")


def normalize_rows(X):
    """Each row of X divided by its L2 norm (a zero row stays zero), without a Python loop."""
    raise NotImplementedError("les normes avec keepdims=True, puis np.where pour la ligne nulle")

In [ ]:
with checked("M07.34 vérifications"):
    import ast
    import inspect
    import textwrap
    problems = []
    rng_t = np.random.default_rng(734)
    X_t = rng_t.normal(size=(7, 5))
    X_t[3] = 0.0                                                     # a zero row
    for p_t in (1, 2, float("inf")):
        expected = np.array([max(abs(v) for v in row) if math.isinf(p_t) else sum(abs(v) ** p_t for v in row) ** (1 / p_t)
                             for row in X_t])
        got = row_norms(X_t, p_t)
        if got is None:
            problems.append("row_norms ne renvoie rien : il manque return")
            break
        got = np.asarray(got)
        if got.shape != (7,):
            problems.append(f"row_norms renvoie la forme {got.shape} : une norme par ligne, de forme (7,)"
                            + (" ; somme sur axis=1, les colonnes" if got.shape == (5,) else ""))
            break
        if not np.allclose(got, expected, atol=1e-12):
            problems.append(f"pour p = {p_t}, row_norms ne donne pas la norme de chaque ligne")
            break
    if not problems:
        unit_t = normalize_rows(X_t)
        if unit_t is None:
            problems.append("normalize_rows ne renvoie rien : il manque return")
        else:
            unit_t = np.asarray(unit_t)
            if unit_t.shape != X_t.shape:
                problems.append(f"normalize_rows renvoie la forme {unit_t.shape}, pas {X_t.shape} : keepdims=True garde "
                                "une colonne pour le broadcasting")
            elif not np.all(np.isfinite(unit_t)):
                problems.append("normalize_rows produit des nan pour la ligne nulle : laisse-la à zéro (np.where)")
            elif not np.allclose(np.delete(np.sqrt(np.sum(unit_t ** 2, axis=1)), 3), 1, atol=1e-12):
                problems.append("les lignes non nulles de normalize_rows n'ont pas la norme 1")
            elif not np.allclose(unit_t[3], 0):
                problems.append("la ligne nulle doit rester nulle")
            elif not np.allclose(unit_t * np.sqrt(np.sum(X_t ** 2, axis=1, keepdims=True)), X_t, atol=1e-12):
                problems.append("normalize_rows doit garder la direction de chaque ligne")
    if not problems:
        for name_t, function_t in [("row_norms", row_norms), ("normalize_rows", normalize_rows)]:
            try:
                tree_t = ast.parse(textwrap.dedent(inspect.getsource(function_t)))
            except (OSError, TypeError, SyntaxError):
                continue
            if any(isinstance(node, (ast.For, ast.While, ast.comprehension)) for node in ast.walk(tree_t)):
                problems.append(f"{name_t} contient une boucle : l'exercice demande de calculer toutes les lignes d'un "
                                "coup (axis, keepdims)")
                break
    if problems:
        print("❌ Ex M07.34 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M07.34 : row_norms et normalize_rows sont justes, sans boucle, ligne nulle comprise.")

**Le temps de calcul** (cette cellule utilise ta fonction).

In [ ]:
with checked("M07.34 temps"):
    import time
    X_big = np.random.default_rng(34).normal(size=(20000, 64))
    start = time.perf_counter()
    loop = np.array([math.sqrt(sum(v * v for v in row)) for row in X_big])
    loop_seconds = time.perf_counter() - start
    start = time.perf_counter()
    fast = np.asarray(row_norms(X_big, 2))
    fast_seconds = time.perf_counter() - start
    print(f"loop: {loop_seconds:.3f} s · row_norms: {fast_seconds:.4f} s · same values: {np.allclose(loop, fast)}")

### Ex M07.35 — Défi : le point qui minimise la somme des distances (non au carré) à 60 villes, à 0,1 % près 🏆 ★★★ ⏱️ 30 min
**Objectif :** trouver numériquement le point qui minimise la somme des distances euclidiennes à des points donnés (la médiane géométrique), et voir qu'il n'est ni le barycentre ni la médiane coordonnée par coordonnée.  
**Prérequis :** fiche §3 et §6 ; Ex M07.26 et M07.31 · **Parcours :** C

Une entreprise veut construire un entrepôt qui dessert 60 villes, et minimiser la **somme des distances** (pas de leurs carrés) de l'entrepôt aux villes, $\sum_i \lVert \mathbf{x}_i - \mathbf{c} \rVert$ (la fonction `total_distance` ci-dessous). Ce point s'appelle la **médiane géométrique**. Il n'a pas de formule : le barycentre minimise la somme des **carrés** (fiche §6), et la médiane coordonnée par coordonnée minimise la somme des distances **L1** (M07.31), mais aucun des deux ne convient ici.

Écris `geometric_median(points)`, qui renvoie le point `[x, y]` (deux nombres). **L'objectif** : une somme des distances à 0,1 % au plus du minimum, en moins de 5 secondes, sur les villes `CITIES_35` et sur quatre autres ensembles de villes. NumPy suffit ; `scipy.optimize` est interdit (c'est lui que la vérification utilise pour trouver le minimum).

La vérification essaie ta fonction sur cinq ensembles de villes, puis vérifie la somme des distances, arrondie à 0,1 km, pour `CITIES_35`.

Dans ta copie : de combien (en %) le barycentre et la médiane coordonnée par coordonnée manquent-ils l'objectif pour `CITIES_35` ? Explique l'idée de ta méthode. Le point trouvé est-il plus proche du groupe de 26 villes ou des deux autres, et pourquoi ?

In [ ]:
def make_cities(seed, angle):
    """60 towns in three groups of 26, 20 and 14, rotated by angle degrees (coordinates in km, rounded to 10 m)."""
    rng = np.random.default_rng(seed)
    centers, sizes = np.array([[0.0, 0.0], [10.0, 1.0], [4.0, 8.0]]), [26, 20, 14]
    towns = np.vstack([rng.normal(c, 1.2, size=(k, 2)) for c, k in zip(centers, sizes)])
    t = np.radians(angle)
    rotation = np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])
    return np.round(towns @ rotation.T + [5.0, 3.0], 2)


def total_distance(points, c):
    """Σ ‖x_i − c‖: the sum of the (non-squared) Euclidean distances from the rows of points to c."""
    return float(np.sum(np.sqrt(np.sum((np.asarray(points, dtype=float) - np.asarray(c, dtype=float)) ** 2, axis=1))))


CITIES_35 = make_cities(735, 20)

In [ ]:
def geometric_median(points):
    """The point [x, y] that minimises the sum of the Euclidean distances to the rows of points (within 0.1 %)."""
    raise NotImplementedError("pars du barycentre, puis améliore : un barycentre pondéré par l'inverse des distances")

In [ ]:
with checked("M07.35 défi"):
    import time
    from scipy.optimize import minimize
    problems, worst_ratio, slowest = [], 0.0, 0.0
    for seed_t, angle_t in [(735, 20), (11, 75), (12, 140), (13, 200), (14, 290)]:
        towns_t = make_cities(seed_t, angle_t)
        best_t = minimize(lambda c: total_distance(towns_t, c), np.median(towns_t, axis=0), method="Nelder-Mead",
                          options={"xatol": 1e-10, "fatol": 1e-10, "maxiter": 20000}).fun
        start_t = time.perf_counter()
        try:
            got_t = geometric_median(towns_t)
        except NotImplementedError:
            raise
        except Exception as exc:  # noqa: BLE001
            problems.append(f"geometric_median lève {type(exc).__name__} ({exc})")
            break
        slowest = max(slowest, time.perf_counter() - start_t)
        if got_t is None or np.shape(got_t) != (2,) or not np.all(np.isfinite(np.asarray(got_t, dtype=float))):
            problems.append("geometric_median doit renvoyer un point du plan : deux nombres finis")
            break
        worst_ratio = max(worst_ratio, total_distance(towns_t, got_t) / best_t)
    if problems:
        print("❌ Ex M07.35 : " + problems[0] + ".")
    elif worst_ratio > 1.001:
        print(f"❌ Ex M07.35 : ton point fait jusqu'à {100 * (worst_ratio - 1):.3f} % de plus que le minimum (objectif : "
              "0,1 % au plus). Le barycentre ne minimise que la somme des carrés : regarde les indices.")
    elif slowest > 5:
        print(f"❌ Ex M07.35 : objectif atteint, mais en {slowest:.1f} s pour un ensemble de villes (objectif : 5 s).")
    else:
        print(f"✅ Ex M07.35 : défi réussi, à {100 * (worst_ratio - 1):.4f} % du minimum au pire, sur cinq ensembles de "
              "villes.")
with checked("M07.35"):
    wbm.check("M07.35", round(total_distance(CITIES_35, geometric_median(CITIES_35)), 1), computed=True)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer un produit scalaire, les normes L1, L2 et L∞ et une distance en dimension $n$ ?
2. Sais-tu calculer un cosinus et une projection, et démontrer l'inégalité de Cauchy-Schwarz ?
3. Sais-tu dire de quel côté d'un hyperplan est un point, et calculer sa distance à l'hyperplan ?
4. Sais-tu pourquoi le barycentre minimise l'inertie, et ce que font k-means, la silhouette et k-means++ ?
5. Sais-tu ce qui change en grande dimension ?

**La suite** : le chapitre 7 de workbookIA (classification, k-means, malédiction de la dimension), puis M08, les tests statistiques.